# 02 — Preprocessing & Feature Engineering

**Objective:** Normalise raw IoT sequences, split into train/val/test without leakage, extract the 1050-dimensional advanced feature vectors, and persist all artefacts.

**Sections:**
1. Load raw data from `data/raw/`
2. Preprocessing pipeline (temporal split + StandardScaler)
3. Feature engineering validation
4. Statistical significance test (Mann-Whitney U)
5. Save feature matrices to `data/features/`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

from src.data.preprocessor import IoTPreprocessor
from src.data.dataset import save_splits, describe_splits, save_baseline_stats
from src.features.feature_pipeline import build_feature_matrix, FEATURE_DIM

sns.set_style("darkgrid")
os.makedirs("../logs", exist_ok=True)
print("Imports OK")

## 1. Load Raw Data

In [ ]:
X = np.load("../data/raw/X.npy")
y = np.load("../data/raw/y.npy")
print(f"Loaded:  X={X.shape}  y={y.shape}")
print(f"Anomaly ratio: {y.mean()*100:.1f}%")

## 2. Preprocessing Pipeline

**Key design decisions:**
- **StandardScaler** (not MinMaxScaler): more robust to outliers (anomalies themselves could inflate max in MinMax)
- Scaler fitted **only on normal training sequences** — never contaminated by anomalies
- **Temporal (ordered) split** — no shuffle to prevent future data leaking into training

In [ ]:
prep = IoTPreprocessor(seq_length=60, n_sensors=50)

splits = prep.full_pipeline(
    X, y,
    save_dir="../data/processed",
    scaler_path="../models/scaler.pkl",
)

X_train = splits["X_train"]
y_train = splits["y_train"]
X_val   = splits["X_val"]
y_val   = splits["y_val"]
X_test  = splits["X_test"]
y_test  = splits["y_test"]

describe_splits(splits)

In [ ]:
# Critical validation
assert np.sum(y_train) == 0, "FAIL: y_train contains anomalies!"
assert X_train.shape[1] == 60 and X_train.shape[2] == 50
print("✅ y_train contains ONLY normals (0 anomalies)")
print(f"✅ Train split is normal-only: {X_train.shape}")

# Visualise normalised distributions
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before normalisation (approximated)
raw_sample = X[y==0][:500].reshape(-1, 50)
axes[0].hist(raw_sample[:, 0], bins=50, color="#FF9800", alpha=0.8)
axes[0].set_title("Sensor 0 — Before Normalisation", fontweight="bold")
axes[0].set_xlabel("Raw value")

# After normalisation
scaled_sample = X_train[:500].reshape(-1, 50)
axes[1].hist(scaled_sample[:, 0], bins=50, color="#4CAF50", alpha=0.8)
axes[1].set_title("Sensor 0 — After StandardScaler", fontweight="bold")
axes[1].set_xlabel("Normalised value")

plt.suptitle("Normalisation Effect on Sensor 0 (Vibration)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("../logs/preprocessing_normalisation.png", dpi=120, bbox_inches="tight")
plt.show()
print(f"Scaled mean ≈ {scaled_sample[:,0].mean():.3f}  std ≈ {scaled_sample[:,0].std():.3f}  (should be ~0, ~1)")

## 3. Feature Engineering

Extracting **1050-dimensional** feature vectors per sequence:
- **Delta (250):** velocity, acceleration, max spike per sensor
- **FFT (450):** fundamental power, harmonics, spectral entropy per sensor
- **Lag (350):** autocorrelation, rolling stats, linear trend per sensor

In [ ]:
print("Building feature matrices (this may take several minutes)…")

# We use ALL training data (both normal + anomaly) for feature matrix
# because XGBoost uses labeled data
X_all_train_raw = np.vstack([splits["X_train"], X_val[:1], X_val[:1]])  # placeholder

# Load the full splits including original (pre-filtered) train
# Actually we need to build features from processed splits
X_train_adv = build_feature_matrix(X_train, verbose=True,
                                    save_path="../data/features/X_train_advanced.npy")

X_val_adv   = build_feature_matrix(X_val, verbose=True,
                                    save_path="../data/features/X_val_advanced.npy")

X_test_adv  = build_feature_matrix(X_test, verbose=True,
                                    save_path="../data/features/X_test_advanced.npy")

print(f"\nFeature matrix shapes:")
print(f"  X_train_adv : {X_train_adv.shape}  (expected: (N, {FEATURE_DIM}))")
print(f"  X_val_adv   : {X_val_adv.shape}")
print(f"  X_test_adv  : {X_test_adv.shape}")

assert X_train_adv.shape[1] == FEATURE_DIM
print(f"\n✅ Feature dimension = {FEATURE_DIM}")

## 4. Statistical Significance Test

**Goal:** Verify that advanced features meaningfully separate normal from anomaly sequences.  
**Method:** Mann-Whitney U test per feature.  
**Criterion:** >50% of features with p-value < 0.05 → feature engineering is effective.

In [ ]:
# Use test set which has mixed labels
X_test_normal  = X_test_adv[y_test == 0]
X_test_anomaly = X_test_adv[y_test == 1]

print(f"Testing {FEATURE_DIM} features (normal n={len(X_test_normal)}, anomaly n={len(X_test_anomaly)})…")

p_values = []
for feat_i in range(FEATURE_DIM):
    _, p = mannwhitneyu(X_test_normal[:, feat_i],
                        X_test_anomaly[:, feat_i],
                        alternative="two-sided")
    p_values.append(p)

p_values = np.array(p_values)
sig_ratio = (p_values < 0.05).mean() * 100

print(f"\nFeatures with p < 0.05: {(p_values < 0.05).sum()} / {FEATURE_DIM}  ({sig_ratio:.1f}%)")
if sig_ratio > 50:
    print(f"✅ Feature engineering EFFECTIVE — {sig_ratio:.1f}% of features are statistically discriminative")
else:
    print(f"⚠️  Only {sig_ratio:.1f}% of features are significant — consider adding more feature types")

# Plot p-value distribution
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(sorted(p_values)[:200], color="#4CAF50", lw=1.5
        , label=f"Sorted p-values (first 200)")
ax.axhline(0.05, color="red", linestyle="--", label="α = 0.05")
ax.set_xlabel("Feature rank (most significant first)")
ax.set_ylabel("p-value")
ax.set_title("Mann-Whitney U p-values: Normal vs Anomaly", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig("../logs/feature_significance.png", dpi=120, bbox_inches="tight")
plt.show()

## 5. Save Baseline Statistics for Drift Monitoring

In [ ]:
save_baseline_stats(
    X_train,
    path="../data/baseline/X_train_baseline_stats.pkl"
)

print("\n✅ All artefacts saved:")
print("   data/processed/   — X_train, X_val, X_test, y_*")
print("   data/features/    — X_*_advanced.npy")
print("   data/baseline/    — X_train_baseline_stats.pkl")
print("   models/scaler.pkl")
print("\n→ NEXT STEP: Run notebook 03_training.ipynb")